# نوفا الصغير — أول دورة تدريب نصية حقيقية (ShamSmall)

هذا الدفتر يشغّل **تدريباً حقيقياً على بيانات عربية حقيقية**، وليس اختباراً تجريبياً. نطاقه محدد بوضوح: **النص فقط** في هذه الجولة الأولى (توليد وفهم نصي). تدريب الصورة والصوت خطوة لاحقة منفصلة تحتاج بيانات صور/صوت حقيقية لم نجمعها بعد — لا داعٍ لتعقيد هذه الجولة الأولى بها.

الكود نفسه مرفوع فعلاً على نفس مستودع GitHub الذي نعمل عليه (`jonsnow-org/Ttbik`، مجلد `ai-system/colab/sham_small`) — هذا الدفتر يسحبه مباشرة من هناك أول ما يعمل، بنفس الطريقة التي كانت تُستخدم مع نوفا سابقاً.

## قبل الضغط على "Run All" — خطوتان فقط مرة واحدة:

1. **أضف رمز وصول (Token) لحسابك على GitHub كـ Kaggle Secret** (لأن المستودع خاص وليس عاماً، فلا يمكن سحبه بلا تصريح):
   - من GitHub: `Settings → Developer settings → Personal access tokens → Generate new token` (يكفي صلاحية `repo` فقط للقراءة).
   - في Kaggle داخل هذا الدفتر: **Add-ons → Secrets → Add a new secret** — اجعل الاسم `GITHUB_TOKEN` والقيمة هي الرمز الذي أنشأته.
2. **فعّل الإنترنت واختر معالج رسومي (GPU):**
   - من القائمة الجانبية: **Settings → Internet → On**.
   - **Settings → Accelerator → GPU T4 x2** (أو أي GPU متاح).

بعد هاتين الخطوتين فقط، اضغط **Run All** ودع الدفتر يعمل من أوله لآخره.


### 1) سحب الكود الحقيقي من GitHub مباشرة

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
# هذا الكود لا يزال على فرع (branch) العمل الحالي، وليس على الفرع الرئيسي main بعد —
# إن دُمج لاحقاً إلى main يمكن حذف --branch هذا بأمان.
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    # يجذب أي تحديث/إصلاح حقيقي جديد تم رفعه على GitHub بعد بدء هذه
    # الجلسة، دون الحاجة لإعادة تشغيل الجلسة أو إعادة استنساخ الكود من الصفر.
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


### 2) تثبيت المكتبات الإضافية غير الموجودة افتراضياً على Kaggle

In [ ]:
# لا نفرض رقم نسخة محدداً عمداً: Kaggle يأتي غالباً بنسخة tokenizers
# حديثة ومتوافقة أصلاً مع مكتبة transformers المثبّتة مسبقاً هناك —
# فرض نسخة قديمة يدوياً يتسبب بتعارض غير ضروري معها (رغم أننا لا
# نستخدم transformers إطلاقاً هنا). هذه الخلية تتحقق أولاً فقط.
try:
    import tokenizers
    print(f"مكتبة tokenizers متوفرة مسبقاً (نسخة {tokenizers.__version__}) — لا حاجة للتثبيت.")
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
    print("تم تثبيت مكتبة tokenizers.")


### 3) جمع بيانات نصية عربية حقيقية

هذه الخلية تسحب (streaming، بدون تنزيل كامل يستهلك المساحة) شريحة حقيقية من **ويكيبيديا العربية** — أكبر مصدر نص عربي نظيف ومجاني متاح مباشرة. يمكنك زيادة `MAX_DOCUMENTS` لاحقاً بعد أن ترى الدفتر يعمل بنجاح من أوله لآخره على عيّنة أصغر.


In [ ]:
from data_acquisition import stream_hf_text_corpus

MAX_DOCUMENTS = 20_000  # ابدأ بعدد معقول للتأكد أن كل شيء يعمل، ثم كبّره في تشغيل لاحق

corpus_dir = "/kaggle/working/corpus/wikipedia_ar"
corpus_files = stream_hf_text_corpus(
    dataset_name="wikimedia/wikipedia",
    config_name="20231101.ar",
    text_field="text",
    output_dir=corpus_dir,
    max_documents=MAX_DOCUMENTS,
)
print(f"عدد ملفات الشحنات (shards) الناتجة: {len(corpus_files)}")


**اختياري:** إذا أردت إضافة معرفة نوفا الحالية الحقيقية (المحادثات وقاعدة المعرفة الموجودة فعلياً في Supabase) كبيانات تدريب إضافية عالية الجودة وباللهجة/الأسلوب نفسه، أضف كـ Kaggle Secrets (من Add-ons → Secrets):
`SUPABASE_URL` و `SUPABASE_SERVICE_ROLE_KEY`، ثم شغّل الخلية التالية. إن لم تُضفها، تجاوز هذه الخلية بأمان — بقية الدفتر يعمل بدونها.


In [ ]:
try:
    from data_acquisition import export_nova_knowledge_to_corpus
    own_corpus_path = "/kaggle/working/corpus/nova_own_knowledge.txt"
    n = export_nova_knowledge_to_corpus(own_corpus_path)
    if n > 0:
        corpus_files.append(own_corpus_path)
    print(f"تمت إضافة {n} فقرة حقيقية من معرفة نوفا الحالية إلى بيانات التدريب.")
except Exception as exc:
    print(f"تم تجاوز هذا المصدر الاختياري (طبيعي إن لم تُضف Kaggle Secrets الخاصة به): {exc}")


### 4) أداة تقسيم النص (Tokenizer)

**مهم جداً عند الاستئناف من جلسة سابقة:** إن كنت أضفت نتاج جلسة سابقة كـ Input (لاستئناف تدريب متوقف)، يجب استخدام **نفس أداة تقسيم النص** التي استُخدمت لتدريب تلك النقطة بالضبط — تدريب أداة جديدة الآن سينتج ربطاً مختلفاً بين الأرقام والكلمات، مما يفسد كل ما تعلّمه النموذج بصمت دون أي خطأ ظاهر. لهذا تبحث الخلية التالية أولاً عن أداة محفوظة من جلسة سابقة، ولا تُدرّب أداة جديدة إلا إن لم تجد شيئاً (أول تشغيل فعلي).


In [ ]:
from pathlib import Path
from text_tokenizer import train_text_tokenizer, ShamTextTokenizer
from model import TEXT_VOCAB_SIZE

previous_tokenizer_files = list(Path("/kaggle/input").rglob("sham_small_tokenizer.json"))
if previous_tokenizer_files:
    tokenizer = ShamTextTokenizer.load(previous_tokenizer_files[0])
    print(f"تم إعادة استخدام أداة تقسيم النص من الجلسة السابقة (vocab={tokenizer.vocab_size}) — "
          f"لضمان توافقها مع نقطة الحفظ المُستأنَفة، بدل تدريب أداة جديدة قد تفسد الأرقام المحفوظة.")
else:
    assert corpus_files, "لا توجد ملفات نصية حقيقية — تحقق من نجاح خلية جمع البيانات أعلاه."
    tokenizer = train_text_tokenizer(corpus_files, vocab_size=TEXT_VOCAB_SIZE)
    print(f"تم تدريب أداة تقسيم نص حقيقية جديدة (vocab={tokenizer.vocab_size}) — أول تشغيل فعلي، لا توجد جلسة سابقة لاستئنافها.")

tokenizer.save("/kaggle/working/sham_small_tokenizer.json")


### 5) بناء بيانات التدريب الفعلية (نافذات نصية بطول ثابت)

هذه الخطوة تحوّل كل النصوص المجمّعة إلى دفعات (batches) جاهزة للتدريب مباشرة.


In [ ]:
import torch
from dataset import TextSequenceDataset

SEQ_LEN = 1024

text_dataset = TextSequenceDataset(corpus_files, tokenizer, seq_len=SEQ_LEN)
print(f"عدد نوافذ التدريب الحقيقية الجاهزة: {len(text_dataset):,} (كل نافذة = {SEQ_LEN} رمزاً)")
assert len(text_dataset) >= 32, (
    "عدد نوافذ التدريب قليل جداً — كبّر MAX_DOCUMENTS في خلية جمع البيانات أعلاه وأعد التشغيل من هناك."
)


### 6) إعداد حجم النموذج

النموذج مبني بالكامل ليتوسع لاحقاً بلا فقدان أي تقدّم مُدرَّب (عبر `expand_model.py`) — لذلك نبدأ بحجم "بداية" آمن يتناسب مع ذاكرة معالج Kaggle المجاني (GPU مجاني عادة 16GB)، ثم نكبّره لاحقاً في جولات قادمة دون الحاجة لإعادة التدريب من الصفر.

- الحجم الافتراضي الكامل للمعمارية (~500 مليون معامل) متاح أيضاً أدناه كخيار — إن كان لديك GPU أقوى.


In [ ]:
from model import ShamSmallConfig, ShamSmall, TOTAL_VOCAB_SIZE

MODEL_SIZE = "starter"  # غيّرها إلى "full" إذا كان لديك GPU أقوى من الـ GPU المجاني القياسي

if MODEL_SIZE == "starter":
    # ~108 مليون معامل حقيقي — آمن على GPU مجاني قياسي (16GB) لأول تشغيل حقيقي
    model_cfg = ShamSmallConfig(
        vocab_size=TOTAL_VOCAB_SIZE, d_model=768, n_layers=12, n_heads=12, n_kv_heads=4,
        mlp_hidden=2048, max_seq_len=SEQ_LEN, use_gradient_checkpointing=True,
    )
else:
    # الحجم الكامل الافتراضي للمعمارية (~500 مليون معامل) — يحتاج GPU أقوى/بدفعات أصغر
    model_cfg = ShamSmallConfig(vocab_size=TOTAL_VOCAB_SIZE, max_seq_len=SEQ_LEN, use_gradient_checkpointing=True)

model = ShamSmall(model_cfg)
n_params = model.count_parameters()
print(f"تم بناء النموذج: {n_params:,} معامل حقيقي (حجم: {MODEL_SIZE}).")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"سيُستخدم للتدريب: {device}" + ("  (تحذير: لا يوجد GPU — تأكد من تفعيله من Settings → Accelerator)" if device == "cpu" else ""))


### 7) استئناف تدريب سابق إن وُجد

إذا كانت هذه ليست أول مرة تشغّل فيها هذا الدفتر (جلسة Kaggle تُغلق تلقائياً بعد 9-12 ساعة)، أضف نتاج (Output) الجلسة السابقة كـ Input جديد لهذا الدفتر، وستكمل هذه الخلية من آخر نقطة حفظ تلقائياً. إن كانت هذه أول مرة، ستبدأ من الصفر تلقائياً بلا أي إجراء إضافي منك.


In [ ]:
from pathlib import Path
from checkpoint import load_checkpoint

start_step = 0
resume_optimizer = None

previous_checkpoints = sorted(
    Path("/kaggle/input").rglob("step_*.pt"),  # يعمل بأي ترتيب مجلدات — لا يشترط اسم "checkpoints" تحديداً
    key=lambda p: int(p.stem.split("_")[1]),
)
if previous_checkpoints:
    last_ckpt = previous_checkpoints[-1]
    model, start_step, _ = load_checkpoint(last_ckpt, map_location=device)
    from train import build_optimizer
    resume_optimizer = build_optimizer(model, lr=3e-4, weight_decay=0.1)
    load_checkpoint(last_ckpt, map_location=device, load_optimizer_into=resume_optimizer)
    print(f"تم استئناف التدريب من نقطة حفظ حقيقية سابقة: {last_ckpt} (الخطوة {start_step:,})")
else:
    print("لم يتم العثور على نقطة حفظ سابقة — سيبدأ التدريب من الصفر (هذا طبيعي في أول تشغيل).")


### 8) قياس السرعة الحقيقية قبل تحديد عدد الخطوات

بدل تخمين عدد الخطوات، نقيس فعلياً كم خطوة في الثانية يحققها معالج Kaggle المخصص لك الآن، ثم نحسب عدد خطوات واقعي يتناسب مع حد الجلسة (9–12 ساعة).


In [ ]:
import time
from train import TrainConfig, build_optimizer, build_lr_scheduler

CALIBRATION_STEPS = 20

calib_batches = [
    torch.stack([text_dataset[i] for i in range(b, b + 2)])
    for b in range(0, min(len(text_dataset) - 2, CALIBRATION_STEPS * 2 * 4), 2)
][: CALIBRATION_STEPS * 4]

assert calib_batches, "لا توجد بيانات كافية للقياس — كبّر MAX_DOCUMENTS في خلية جمع البيانات."

model.to(device)
model.train()
_calib_optimizer = resume_optimizer or build_optimizer(model, lr=3e-4, weight_decay=0.1)

t0 = time.time()
steps_done = 0
for batch in calib_batches:
    batch = batch.to(device)
    _, loss = model(batch, labels=batch)
    loss.backward()
    _calib_optimizer.step()
    _calib_optimizer.zero_grad()
    steps_done += 1
    if steps_done >= CALIBRATION_STEPS:
        break
elapsed = time.time() - t0
steps_per_second = steps_done / elapsed

# قيمة ثابتة وآمنة تحت حد طول الجلسة الواحدة في Kaggle (لوحظ فعلياً عند
# ~12 ساعة قبل القتل القسري) — وليست مرتبطة برصيدك الأسبوعي المتبقي، لأن
# هذه الخلية قد تعمل الآن بلا إشراف بشري (تشغيل مجدول تلقائي، انظر الخلية
# 11 أدناه)، فلا يوجد من يعدّلها يدوياً قبل كل جلسة. إن كان رصيدك الأسبوعي
# المتبقي أقل من هذه الساعات، ستنتهي الجلسة ببساطة بسبب نفاد الرصيد نفسه
# (وليس بسبب هذا الرقم) — وهذا آمن تماماً بفضل نظام الاستئناف: الجلسة
# المجدولة القادمة (بعد تجدّد الرصيد الأسبوعي) تكمل من حيث توقفت تلقائياً.
# غيّرها يدوياً فقط إن كنت تراقب الجلسات بنفسك دون جدولة تلقائية.
MAX_TRAINING_HOURS = 8.5
realistic_steps_for_session = int(steps_per_second * MAX_TRAINING_HOURS * 3600 * 0.85)  # هامش أمان 15% إضافي

print(f"سرعة حقيقية مقاسة الآن: {steps_per_second:.3f} خطوة/ثانية على {device}")
print(f"عدد خطوات واقعي يمكن إنجازه ضمن {MAX_TRAINING_HOURS} ساعة (بهامش أمان): {realistic_steps_for_session:,} خطوة")


### 9) التدريب الحقيقي

عدد الخطوات أدناه محسوب تلقائياً من القياس الحقيقي أعلاه — لا حاجة لتعديله يدوياً، لكن يمكنك تصغيره لو أردت تشغيلاً أقصر للتجربة أولاً (مثلاً بوضع `TOTAL_STEPS = 500`).

**حماية حقيقية إضافية:** حتى لو كان تقدير الخطوات أعلاه متفائلاً أكثر من اللازم، `max_wall_clock_seconds` أدناه يجعل التدريب **يوقف نفسه فعلياً** ويحفظ نقطة حفظ حقيقية بمجرد الاقتراب من `MAX_TRAINING_HOURS` — بدل انتظار Kaggle ليقتل الجلسة بالقوة بلا أي حفظ (كما حدث فعلياً من قبل).


In [ ]:
TOTAL_STEPS = max(realistic_steps_for_session, 200)
num_windows = len(text_dataset) - (len(text_dataset) % 4)
epochs_needed = -(-TOTAL_STEPS // (num_windows // 4))  # للطباعة فقط، تقريب لأعلى
print(f"عدد نوافذ التدريب الحقيقية المتاحة: {num_windows:,} — ستُستخدم عبر {epochs_needed} دورة/دورات (epochs) لإنجاز {TOTAL_STEPS:,} خطوة.")

def _batch_iterator():
    # تبني كل دفعة عند الحاجة فقط بدل تجهيز كل دفعات التدريب مسبقاً في
    # قائمة واحدة ضخمة (وتكرارها) — التجهيز المسبق كان يُبقي نسخة كاملة
    # إضافية من كل نوافذ التدريب في RAM طوال الجلسة، فوق النسخة التي
    # يحتفظ بها text_dataset نفسه أصلاً، وهو ما أسقط جلسة Colab فعلياً
    # بعد استنفاد كل RAM المتاح (حادثة حقيقية، 2026-09-15).
    while True:
        for b in range(0, num_windows, 4):
            yield torch.stack([text_dataset[i] for i in range(b, b + 4)])

import itertools
batches = itertools.islice(_batch_iterator(), TOTAL_STEPS)

train_cfg = TrainConfig(
    seq_len=SEQ_LEN,
    batch_size=4,
    grad_accum_steps=4,          # حجم دفعة فعلي = 16
    lr=3e-4,
    warmup_steps=max(50, TOTAL_STEPS // 100),
    total_steps=start_step + TOTAL_STEPS,
    checkpoint_dir="/kaggle/working/checkpoints",
    checkpoint_every=200,
    log_every=20,
    max_wall_clock_seconds=MAX_TRAINING_HOURS * 3600,
)

from train import train
loss_history = train(
    model, batches, train_cfg, device=device,
    start_step=start_step, resume_optimizer=resume_optimizer or _calib_optimizer,
)

print(f"\nانتهى التدريب على {len(loss_history):,} خطوة حقيقية.")
print(f"متوسط الخسارة (loss) في أول 10 خطوات: {sum(loss_history[:10]) / min(10, len(loss_history)):.4f}")
print(f"متوسط الخسارة (loss) في آخر 10 خطوات: {sum(loss_history[-10:]) / min(10, len(loss_history)):.4f}")


### 10) حفظ النتيجة النهائية

آخر نقطة حفظ محفوظة أصلاً تلقائياً أثناء التدريب في `/kaggle/working/checkpoints/`. هذه الخلية تحفظ أيضاً نسخة أخيرة صريحة + أداة تقسيم النص، بحيث كل ما تحتاجه لاستئناف التدريب أو لتشغيل النموذج عبر `serve.py` موجود في نتاج هذه الجلسة (Output) تلقائياً — لا حاجة لتنزيل أي شيء يدوياً؛ بعد انتهاء تشغيل الدفتر اضغط **Save Version** ليصبح هذا الناتج قابلاً لإضافته كـ Input لجلسة تدريب أو خدمة قادمة.


In [ ]:
from checkpoint import save_checkpoint

final_step = start_step + len(loss_history)
save_checkpoint("/kaggle/working/checkpoints/final.pt", model, final_step)
print(f"تم حفظ النقطة النهائية عند الخطوة {final_step:,} في /kaggle/working/checkpoints/final.pt")
print("أداة تقسيم النص محفوظة في /kaggle/working/sham_small_tokenizer.json")
print("\nلا تنسَ: اضغط الآن Save Version أعلى الصفحة حتى يُحفظ كل هذا كـ Output دائم لهذه الجلسة.")


### 11) نشر نقطة الحفظ تلقائياً كنسخة جديدة على Kaggle Dataset — لسلسلة تدريب بلا تدخل يدوي

**هذه الخلية هي ما يجعل التدريب يُكمل نفسه بنفسه عبر عدة جلسات، دون أن تفتح الهاتف أو تنشئ Dataset يدوياً كل مرة.**

آلية العمل:
1. تنشر هذه الخلية آخر نقطة حفظ + أداة تقسيم النص كـ **نسخة جديدة (version)** من نفس الـ Dataset (`nova-small-checkpoint`) الذي أنشأته يدوياً أول مرة — عبر Kaggle API مباشرة من داخل الكود.
2. إن كان هذا الدفتر **مُجدولاً للعمل تلقائياً** (Schedule a notebook to run)، فالجلسة القادمة ستجد هذا الـ Dataset بآخر نسخة منه مرفقاً تلقائياً كـ Input (بشرط عدم تثبيته على إصدار معيّن)، فتلتقط نقطة الحفظ الجديدة تلقائياً بلا أي تدخل.

**إعداد لمرة واحدة فقط قبل استخدام هذه الخلية:**
1. اذهب إلى https://www.kaggle.com/settings → قسم **API** → **Create New Token** — سيُنزَّل ملف `kaggle.json` يحوي `username` و `key`.
2. في هذا الدفتر: **Add-ons → Secrets → Add a new secret** مرتين:
   - الاسم `KAGGLE_USERNAME` والقيمة هي `username` من الملف.
   - الاسم `KAGGLE_KEY` والقيمة هي `key` من الملف.
3. تأكد أن اسم الـ Dataset الذي أنشأته يدوياً هو فعلاً `nova-small-checkpoint` (كما هو أدناه)، أو غيّر `DATASET_SLUG` ليطابق اسمك الحقيقي.

**بعد إعداد هذا مرة واحدة:** من القائمة الجانبية اضغط **"Schedule a notebook to run"** واختر تكراراً (مثلاً كل يوم) — ومن الآن فصاعداً لن تحتاج فتح Kaggle يدوياً إطلاقاً؛ فقط راقب تقدّم الخسارة (loss) بين حين وآخر.

In [ ]:
import json as _json
import os as _os
import subprocess as _subprocess
import shutil as _shutil
from pathlib import Path as _Path
from kaggle_secrets import UserSecretsClient as _UserSecretsClient

_subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=False)

KAGGLE_USERNAME = _UserSecretsClient().get_secret("KAGGLE_USERNAME")
KAGGLE_KEY = _UserSecretsClient().get_secret("KAGGLE_KEY")
DATASET_SLUG = f"{KAGGLE_USERNAME}/nova-small-checkpoint"  # يجب أن يطابق اسم Dataset الذي أنشأته يدوياً

_os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
_os.environ["KAGGLE_KEY"] = KAGGLE_KEY

upload_dir = _Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    _shutil.rmtree(upload_dir)
(upload_dir / "checkpoints").mkdir(parents=True)
for ckpt in _Path("/kaggle/working/checkpoints").glob("*.pt"):
    _shutil.copy2(ckpt, upload_dir / "checkpoints" / ckpt.name)
_shutil.copy2("/kaggle/working/sham_small_tokenizer.json", upload_dir / "sham_small_tokenizer.json")

metadata = {"title": "nova-small-checkpoint", "id": DATASET_SLUG, "licenses": [{"name": "unknown"}]}
(upload_dir / "dataset-metadata.json").write_text(_json.dumps(metadata))

result = _subprocess.run(
    ["kaggle", "datasets", "version", "-p", str(upload_dir), "-m", f"auto-update at step {final_step:,}", "-r", "skip"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("تنبيه: فشل نشر النسخة الجديدة تلقائياً — لا يؤثر هذا على نقطة الحفظ نفسها (لا تزال محفوظة في Output هذه الجلسة بأمان)، "
          "لكن يعني أن الجلسة القادمة لن تلتقطها تلقائياً. تحقق من صحة KAGGLE_USERNAME / KAGGLE_KEY ومن اسم DATASET_SLUG.")
    print(result.stderr)
else:
    print(f"تم نشر نقطة الحفظ عند الخطوة {final_step:,} كنسخة جديدة من {DATASET_SLUG} — الجلسة القادمة (المجدولة) ستلتقطها تلقائياً.")
